# RALPH BLASTP inference

This notebook is the user-facing entry point for the RALPH BLAST baseline.
Given a LuxR-homolog query sequence, it runs BLASTP against the RALPH
database and visualizes the top-ranked ligands using the ENSEMBLE method.

## Before running

1. From the repo root, run `uv sync` to install the `ralph` package and
   its dependencies (ships in-tree under `src/ralph/`).
2. Install BLAST+ (`blastp` on PATH). See `INSTALLATION.md`.
3. Build the RALPH database. See `baselines/ralph_blast/README.md`. By
   default this notebook looks for the database at
   `../baselines/ralph_blast/ralph_database` **relative to the working
   directory the Jupyter kernel was launched in**. Launch Jupyter from
   the `applications/` directory (`cd applications && jupyter notebook
   RALPH_blastp_inference.ipynb`) for the default to resolve correctly,
   or set the `RALPH_DATABASE` environment variable to an absolute path
   if you want to run Jupyter from elsewhere.

## Running

Change the `query_sequence` variable in the next cell, then run all cells.

In [ ]:
#@title Change the variables below, then hit `Runtime` -> `Run all`
query_sequence = 'MQHWLDKLTDLAAIQGDECILKDGLADLAEHFGFTGYAYLHIQHKHTIAVTNYHRDWRSAYFENNFDKLDPVVKRAKSRKHVFAWSGEQERSRLSKEERAFYAHAADFGIRSGITIPIKTANGSMSMFTLASERPAIDLDREIDAAAAAGAVGQLHARISFLQTTPTVEDAAWLDPKEATYLRWIAVGMTMEEVADVEGVKYNSVRVKLREAMKRFDVRSKAHLTALAIRRKLI' #@param {type:"string"}
#@markdown  Change the variable below to change how many sequence hits to use from BLASTP
top_k_BLASTP_hits = 50 #@param {type:"raw"}
top_k = top_k_BLASTP_hits

In [ ]:
#@title Environment check and database path

import os
import shutil

# --- ralph availability ---
# ralph ships in this repository under src/ralph/ and is installed by
# `uv sync` from the repo root. If this import fails, run `uv sync`.
from ralph import (
    blastp_query,
    aggregate_rankings,
    clustered_rank_plot,
    load_scores,
    load_ligands,
    interactive_blastp_results,
    interactive_alignment_viewer,
)
import pandas as pd
import numpy as np

# --- BLAST+ availability ---
if shutil.which("blastp") is None:
    raise RuntimeError(
        "blastp not on PATH. Install BLAST+ and re-run. "
        "See INSTALLATION.md in the repo root."
    )

# --- Database path ---
# Default: the baseline's working directory relative to this notebook.
# Override with the RALPH_DATABASE environment variable if your database
# lives elsewhere.
database_path = os.environ.get(
    "RALPH_DATABASE",
    os.path.abspath(os.path.join(os.getcwd(), "..", "baselines", "ralph_blast", "ralph_database")),
)
if not os.path.isdir(database_path):
    raise FileNotFoundError(
        f"RALPH database directory not found at {database_path}. "
        f"See baselines/ralph_blast/README.md for build instructions, or "
        f"set RALPH_DATABASE to an existing database directory."
    )

df = load_scores(database_path=database_path)
ligand_df = load_ligands(database_path=database_path)

In [ ]:
#@title Running BLASTP query on RALPH and Displaying Aggregated Rankings
# All helper functions are now imported from the ralph package
# (blastp_query, aggregate_rankings, clustered_rank_plot)

# database_path is already defined in the setup cell above.
db_path = os.path.join(database_path, "ralph")

hits = blastp_query(
    query_seq=query_sequence,
    db_path=db_path,
    max_hits=top_k,
)

hits = hits.sort_values("bitscore", ascending=False).head(top_k)

scores = df[df.luxr_name.isin(hits.sseqid)]

In [ ]:
#@title Interactive BLASTP Results Visualization
# Select a hit from the dropdown to view:
# 1. Sequence alignment plot
# 2. Top 58 molecules (visualized if ligand_df is provided)
# 3. Clustered rankings plot with top_k=58

interactive_blastp_results(
    hits_df=hits,
    scores_df=df,
    query_sequence=query_sequence,
    ligand_df=ligand_df
)